# DevTitans — QLoRA pilot for notification summarization
**Prepared experiment; no completed training or quality improvement is claimed.**

Use a free Colab GPU only. No paid jobs, automatic subscriptions or Hub uploads are invoked. GPU availability is not guaranteed. This pilot adapts **Gemma 3 1B IT**, separately from the Android PoC's Gemma 4 E2B. It does not establish LiteRT export compatibility.

Before running, review and accept the model's terms yourself at https://huggingface.co/google/gemma-3-1b-it if you agree. Authenticate inside this runtime; never place a token in a notebook cell. A Colab session does not inherit your Windows login.

This bounded pilot uses 512 training examples and 16 validation examples. The test split remains untouched. Dataset templates can overlap semantically across splits; improved validation results would still require independent scenarios and human factuality review. See `docs/EVIDENCE.md` for authorship.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==5.22.0', 'peft==0.21.0', 'accelerate', 'bitsandbytes', 'datasets'])
# Keep the CUDA-enabled PyTorch supplied by the notebook runtime.


In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a free GPU runtime first; do not buy compute.'
assert torch.cuda.get_device_properties(0).total_memory >= 14 * 1024**3, 'This pilot expects roughly 16 GiB GPU memory.'
print(torch.cuda.get_device_name(0))
from huggingface_hub import notebook_login
notebook_login()  # Interactive credentials; never print or save tokens in cells.


In [ ]:
import hashlib, json, time, importlib.metadata as metadata
from pathlib import Path
from datasets import load_dataset
from huggingface_hub import HfApi
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          TrainingArguments, Trainer, DataCollatorForSeq2Seq, set_seed)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
BASE = 'google/gemma-3-1b-it'
DATA = 'phbg/smart-notifications-synthetic'
DATA_REV = '7cb29736e49ed7dd4de6b31bf5d3ff033fa9e316'
MODEL_REV = HfApi().model_info(BASE).sha
SEED = 42
OUT = Path('devtitans-qlora-pilot')
assert not OUT.exists(), 'Use a fresh output directory to preserve the previous experiment.'
OUT.mkdir()
set_seed(SEED)
raw = load_dataset(DATA, revision=DATA_REV)
tokenizer = AutoTokenizer.from_pretrained(BASE, revision=MODEL_REV)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
PROMPT = ('Resuma as notificações em português do Brasil, em um parágrafo de no máximo 25 palavras. '
          'Preserve nomes, datas, horários, prazos e ações atuais. Ignore repetições, ruído e fatos corrigidos. '
          'Não invente informações. Responda somente com o resumo, sem Markdown.')
def message(text):
    return PROMPT + '\n<mensagens>\n' + text + '\n</mensagens>'
def encode(row):
    user = [{'role':'user', 'content':message(row['input'])}]
    prompt_ids = tokenizer.apply_chat_template(user, tokenize=True, add_generation_prompt=True)
    full_ids = tokenizer.apply_chat_template(user + [{'role':'assistant','content':row['output']}], tokenize=True)
    assert full_ids[:len(prompt_ids)] == prompt_ids, 'Unexpected template boundary'
    return {'input_ids':full_ids, 'attention_mask':[1]*len(full_ids),
            'labels':[-100]*len(prompt_ids) + full_ids[len(prompt_ids):]}
def prepare(split, n):
    selected = raw[split].shuffle(seed=SEED).select(range(min(n,len(raw[split]))))
    encoded = selected.map(encode, remove_columns=selected.column_names)
    kept = encoded.filter(lambda x: len(x['input_ids']) <= 768 and any(y != -100 for y in x['labels']))
    assert len(kept) > 0
    print(split, 'selected', len(selected), 'retained', len(kept))
    return kept
train = prepare('train', 512)
validation = prepare('validation', 16)
eval_rows = [r for r in raw['validation'].shuffle(seed=SEED) if len(tokenizer.apply_chat_template([{'role':'user','content':message(r['input'])}], tokenize=True, add_generation_prompt=True)) + 48 <= 768][:16]
assert eval_rows, 'No eligible validation examples'
model = AutoModelForCausalLM.from_pretrained(BASE, revision=MODEL_REV,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16),
    device_map={'':0}, torch_dtype=torch.float16)


In [ ]:
def generate_rows():
    model.eval()
    records=[]
    for row in eval_rows:
        inputs=tokenizer.apply_chat_template([{'role':'user','content':message(row['input'])}],
            tokenize=True, add_generation_prompt=True, return_tensors='pt', return_dict=True).to('cuda')
        assert inputs['input_ids'].shape[-1] + 48 <= 768, 'Evaluation input exceeds fixed budget'
        with torch.inference_mode():
            result=model.generate(**inputs, max_new_tokens=48, do_sample=False,
                pad_token_id=tokenizer.pad_token_id)
        prediction=tokenizer.decode(result[0,inputs['input_ids'].shape[-1]:],skip_special_tokens=True).strip()
        records.append({'input':row['input'],'reference':row['output'],'prediction':prediction})
    return records
baseline=generate_rows()
(OUT/'baseline.json').write_text(json.dumps(baseline,ensure_ascii=False,indent=2),encoding='utf-8')
model=prepare_model_for_kbit_training(model)
model=get_peft_model(model,LoraConfig(r=8,lora_alpha=16,lora_dropout=0.05,bias='none',
    task_type='CAUSAL_LM',target_modules=['q_proj','v_proj']))
model.config.use_cache=False
model.print_trainable_parameters()
before={n:p.detach().cpu().clone() for n,p in model.named_parameters() if p.requires_grad}
assert before, 'No trainable adapter weights'
args=TrainingArguments(output_dir=str(OUT/'checkpoints'),max_steps=64,
    per_device_train_batch_size=1,per_device_eval_batch_size=1,gradient_accumulation_steps=8,
    learning_rate=1e-4,warmup_steps=4,fp16=True,bf16=False,gradient_checkpointing=True,
    logging_steps=8,eval_strategy='steps',eval_steps=32,save_strategy='steps',save_steps=32,
    save_total_limit=2,report_to='none',seed=SEED,data_seed=SEED,
    optim='adamw_torch',push_to_hub=False)
trainer=Trainer(model=model,args=args,train_dataset=train,eval_dataset=validation,
    data_collator=DataCollatorForSeq2Seq(tokenizer,model=None,padding=True,label_pad_token_id=-100),
    processing_class=tokenizer)
torch.cuda.reset_peak_memory_stats()
started=time.perf_counter()
result=trainer.train()
elapsed=time.perf_counter()-started
changed=sum(not torch.equal(before[n],p.detach().cpu()) for n,p in model.named_parameters() if n in before)
assert changed > 0 and trainer.state.global_step > 0, 'No verified parameter updates'
model.save_pretrained(OUT/'adapter',safe_serialization=True)
tokenizer.save_pretrained(OUT/'adapter')
model.config.use_cache=True
model.gradient_checkpointing_disable()
adapted=generate_rows()
(OUT/'adapted.json').write_text(json.dumps(adapted,ensure_ascii=False,indent=2),encoding='utf-8')
def automatic_checks(rows):
    return {'n':len(rows),'nonempty':sum(bool(r['prediction']) for r in rows),
        'within_25_words':sum(0 < len(r['prediction'].split()) <= 25 for r in rows)}
report={'status':'pilot_training_completed_not_quality_validated','base_model':BASE,
 'base_revision':MODEL_REV,'dataset':DATA,'dataset_revision':DATA_REV,'seed':SEED,
 'steps':trainer.state.global_step,'changed_trainable_tensors':changed,
 'train_rows':len(train),'validation_rows':len(validation),'elapsed_seconds':elapsed,
 'peak_gpu_memory_bytes':torch.cuda.max_memory_allocated(), 'training_metrics':result.metrics,
 'baseline_checks':automatic_checks(baseline),'adapted_checks':automatic_checks(adapted),
 'packages':{p:metadata.version(p) for p in ['torch','transformers','peft','datasets','accelerate','bitsandbytes']},
 'gpu':torch.cuda.get_device_name(0), 'test_split_used':False,
 'warning':'Automatic word counts do not measure factuality. Independent reviewed examples still required.'}
(OUT/'report.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
(OUT/'training-config.json').write_text(args.to_json_string(),encoding='utf-8')
print(json.dumps(report,indent=2))


## Inspect and save before the free session ends
Compare baseline.json and adapted.json without cherry-picking. Check current dates/actions and unsupported additions manually. This pilot does not establish improvement merely because loss decreased. Do not retune using a future independent test set. Publishing a model card requires the actual report and evaluation; this notebook does not upload anything automatically.

Save the archive below. It contains synthetic-data outputs, adapter weights, training configuration and checkpoints; it must not include account tokens. Cloud sessions can expire. Re-running training in a new runtime is not guaranteed to be bit-identical.


In [ ]:
import shutil
archive=shutil.make_archive(str(OUT), 'zip', OUT)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Download from the notebook file browser:', archive)
